# Modeling Prep

## 1. Konfigurasi

In [1]:
import json
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd


def find_base_dir(start=None) -> Path:
    """Cari root repo — folder pertama ke atas yang berisi `dataset/csv/`."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset" / "csv").is_dir():
            return candidate
    raise RuntimeError(f"Root repo tidak ditemukan dari {start}")


BASE_DIR = find_base_dir()

MODEL_READY_DIR = str(BASE_DIR / "dataset/model_ready")
FEATURED_FILE = str(BASE_DIR / "dataset/model_ready/featured.parquet")
MODEL_INPUT_FILE = str(BASE_DIR / "dataset/model_ready/model_input.parquet")
EVENT_ITEMS_FILE = str(BASE_DIR / "dataset/event_driven_items.csv")
CATEGORY_MAPPING_FILE = str(BASE_DIR / "dataset/model_ready/category_mapping.json")
SCALER_FILE = str(BASE_DIR / "dataset/model_ready/scaler_params.json")

PAIR_COLS = ["Kode Barang", "Nama Cabang"]


SEGMENT_COL = "segment_id"


SEGMENT_COLS = PAIR_COLS + [SEGMENT_COL]


DATE_COL = "Tanggal"


TRAIN_TARGET_COL = "target_lead_time_cumulative_capped"


EVAL_TARGET_COL = "target_lead_time_cumulative"


LOOKBACK = 28

# Desember 2025 adalah test set terkunci — tidak ada fold walk-forward yang boleh menyentuhnya.
TEST_START = pd.Timestamp("2025-12-01")



print(f"BASE_DIR         = {BASE_DIR}")
print(f"FEATURED_FILE    = {Path(FEATURED_FILE).name}  "
      f"({'ada' if Path(FEATURED_FILE).exists() else 'HILANG'})")
print(f"MODEL_INPUT_FILE = {Path(MODEL_INPUT_FILE).name}")
print(f"TEST_START       = {TEST_START.date()}  (Desember 2025 tidak masuk fold mana pun)")
print(f"target latih     = {TRAIN_TARGET_COL}")
print(f"target evaluasi  = {EVAL_TARGET_COL}")

BASE_DIR         = /Users/ramapdp/Project/Personal/forecast-scm
FEATURED_FILE    = featured.parquet  (ada)
MODEL_INPUT_FILE = model_input.parquet
TEST_START       = 2025-12-01  (Desember 2025 tidak masuk fold mana pun)
target latih     = target_lead_time_cumulative_capped
target evaluasi  = target_lead_time_cumulative


## 2. Flag item event-driven

In [2]:
def load_event_items(path: str = EVENT_ITEMS_FILE) -> pd.DataFrame:
    return pd.read_csv(path, sep=";", encoding="utf-8-sig")


def add_event_flag(
    df: pd.DataFrame,
    event_items_df: pd.DataFrame,
    item_col: str = "Kode Barang",
) -> pd.DataFrame:
    """Attach the per-SKU is_event_driven flag from event_driven_items.csv.

    Raises rather than defaulting when a SKU is absent from the list: a new SKU
    appearing in a monthly refresh must be classified by the data owner, not
    silently assumed non-event.
    """
    result = df.copy()
    flags = (
        event_items_df.set_index(item_col)["is_event_driven"]
        .astype(str).str.strip().str.lower().eq("true")
    )
    mapped = result[item_col].map(flags)
    if mapped.isna().any():
        missing = sorted(result.loc[mapped.isna(), item_col].unique())
        raise ValueError(
            f"SKU tanpa entri di {EVENT_ITEMS_FILE}: {missing}. "
            "Tambahkan barisnya dan minta klasifikasi dari data owner."
        )
    result["is_event_driven"] = mapped.astype(bool)
    return result

## 3. Segmentasi permintaan (ADI / CV²)

In [3]:
ADI_THRESHOLD = 1.32


CV2_THRESHOLD = 0.49


def compute_pair_demand_stats(
    df: pd.DataFrame,
    cutoff: pd.Timestamp = TEST_START,
    qty_col: str = "Kuantitas",
    pair_cols: list = None,
    date_col: str = DATE_COL,
) -> pd.DataFrame:
    """ADI and CV2 per pair, computed from the training period only.

    Deriving these from the full series would leak post-cutoff behaviour into
    a feature the model trains on.
    """
    pair_cols = pair_cols or PAIR_COLS
    train = df[df[date_col] < cutoff]
    grouped = train.groupby(pair_cols, observed=True)[qty_col]

    n_days = grouped.size()
    n_nonzero = grouped.apply(lambda s: int((s > 0).sum()))
    nz_mean = grouped.apply(lambda s: s[s > 0].mean())
    nz_std = grouped.apply(lambda s: s[s > 0].std(ddof=0))

    adi = n_days / n_nonzero.replace(0, np.nan)
    cv2 = (nz_std / nz_mean.replace(0, np.nan)) ** 2

    return pd.DataFrame({"adi": adi, "cv2": cv2.fillna(0.0)})


def _segment_label(adi: float, cv2: float) -> str:
    if pd.isna(adi):
        # Never moved during the training period — treat as the hardest case.
        return "lumpy"
    if adi < ADI_THRESHOLD:
        return "smooth" if cv2 < CV2_THRESHOLD else "erratic"
    return "intermittent" if cv2 < CV2_THRESHOLD else "lumpy"


def classify_pairs(
    df: pd.DataFrame,
    cutoff: pd.Timestamp = TEST_START,
    qty_col: str = "Kuantitas",
    pair_cols: list = None,
    date_col: str = DATE_COL,
) -> pd.DataFrame:
    """Klasifikasikan setiap pasangan ke salah satu dari 4 segmen Syntetos-Boylan.

    Matriks klasifikasi (ADI > 1.32 → intermittent, CV² > 0.49 → erratic):
      ADI rendah, CV² rendah  →  smooth
      ADI rendah, CV² tinggi  →  erratic
      ADI tinggi, CV² rendah  →  intermittent
      ADI tinggi, CV² tinggi  →  lumpy
    """
    pair_cols = pair_cols or PAIR_COLS
    stats = compute_pair_demand_stats(
        df, cutoff=cutoff, qty_col=qty_col, pair_cols=pair_cols, date_col=date_col
    )
    labels = stats.apply(lambda r: _segment_label(r["adi"], r["cv2"]), axis=1)
    labels.name = "demand_segment"

    result = df.copy()
    result["demand_segment"] = (
        result.set_index(pair_cols).index.map(labels).astype(object)
    )
    result["demand_segment"] = result["demand_segment"].fillna("lumpy")
    return result

## 4. Purging & pembagian fold walk-forward

In [4]:
def lookahead_safe_mask(
    df: pd.DataFrame,
    boundary: pd.Timestamp,
    date_col: str = "Tanggal",
    lead_time_col: str = "lead_time_days",
) -> pd.Series:
    """True for rows whose whole target window stays strictly before `boundary`.

    A null lead time is safe: without one there is no lead-time target for the
    boundary to contaminate. Rows dated on or after the boundary come out
    False, which is harmless — callers combine this with their own date filter
    and never train on those rows anyway.
    """
    lead_time = pd.to_timedelta(df[lead_time_col].fillna(0), unit="D")
    return df[date_col] + lead_time < boundary


FOLD_STARTS = [
    pd.Timestamp("2025-07-01"),
    pd.Timestamp("2025-08-01"),
    pd.Timestamp("2025-09-01"),
    pd.Timestamp("2025-10-01"),
    pd.Timestamp("2025-11-01"),
]


def assign_folds(
    df: pd.DataFrame,
    fold_starts: list = None,
    date_col: str = DATE_COL,
) -> pd.DataFrame:
    """Beri nomor fold ke setiap baris berdasarkan bulan kalender.

    Nomor fold dimulai dari 1 sesuai urutan FOLD_STARTS. Baris di luar semua
    jendela fold mendapat NaN — baris ini tidak dipakai dalam walk-forward.
    """
    fold_starts = fold_starts or FOLD_STARTS
    result = df.copy()
    result["fold_id"] = np.nan
    for number, start in enumerate(fold_starts, start=1):
        end = start + pd.offsets.MonthBegin(1)
        in_month = (result[date_col] >= start) & (result[date_col] < end)
        result.loc[in_month, "fold_id"] = float(number)
    return result


def fold_train_mask(
    df: pd.DataFrame,
    fold_id: int,
    fold_starts: list = None,
    date_col: str = DATE_COL,
    purge: bool = True,
) -> pd.Series:
    """Rows usable for training fold `fold_id` — strictly before its month.

    Purging drops the last few days before the fold boundary, whose lead-time
    label is summed partly over the validation month itself. Same reasoning as
    prepare_forecast_data.split_train_test.
    """
    fold_starts = fold_starts or FOLD_STARTS
    if not 1 <= fold_id <= len(fold_starts):
        raise ValueError(f"fold_id harus 1..{len(fold_starts)}, dapat {fold_id}")
    boundary = fold_starts[fold_id - 1]
    mask = df[date_col] < boundary
    if purge and "lead_time_days" in df.columns:
        mask &= lookahead_safe_mask(df, boundary, date_col=date_col)
    return mask

## 5. Encoding kolom kategorikal

In [5]:
CATEGORICAL_COLS = [
    "Kode Barang",
    "Nama Cabang",
    "Kategori Barang",
    "kota",
    "hari_pengiriman",
    "branch_volume_tier",
    "demand_segment",
]


UNKNOWN_TOKEN = "<UNKNOWN>"


UNKNOWN_INDEX = 0


def build_category_mapping(
    df: pd.DataFrame,
    cutoff: pd.Timestamp = TEST_START,
    cols: list = None,
    date_col: str = DATE_COL,
    existing: Optional[dict] = None,
) -> dict:
    """Fit value -> index maps from the training period only.

    Fitting on train only is a correctness requirement, not tidiness: a branch
    that only appears after the cutoff must not enter the mapping at all.

    That alone does not survive a refresh, though. When the cutoff moves
    forward, values that used to sit in the test period cross into the
    training period and join the mapping for real -- and re-sorting the whole
    set renumbers every value sorting after them. Measured on this dataset:
    six new SKUs entering training shift the index of 32 of the 70 existing
    ones, which silently invalidates any model already trained on the old
    numbering. Pass `existing` (normally the previously saved mapping) to keep
    every index already handed out and append new values after the highest one.
    """
    cols = cols or CATEGORICAL_COLS
    existing = existing or {}
    train = df[df[date_col] < cutoff] if date_col in df.columns else df
    mapping = {}
    for col in cols:
        values = sorted(str(v) for v in train[col].dropna().unique())
        # Retired values keep their index: freeing it up for a new value would
        # point an already-trained model at the wrong category.
        previous = dict(existing.get(col) or {})
        previous.setdefault(UNKNOWN_TOKEN, UNKNOWN_INDEX)
        next_index = max(previous.values()) + 1
        for value in values:
            if value not in previous:
                previous[value] = next_index
                next_index += 1
        mapping[col] = previous
    return mapping


def encode_categoricals(
    df: pd.DataFrame,
    mapping: dict,
    cols: list = None,
) -> pd.DataFrame:
    """Konversi nilai kategorikal menjadi indeks integer menggunakan mapping yang tersimpan.

    Nilai yang tidak dikenali (mis. cabang baru saat refresh) dipetakan ke
    UNKNOWN_INDEX (0), bukan di-raise sebagai error, supaya pipeline tidak
    berhenti hanya karena ada cabang baru.
    """
    cols = cols or CATEGORICAL_COLS
    result = df.copy()
    for col in cols:
        result[f"{col}_idx"] = (
            result[col].astype(str).map(mapping[col]).fillna(UNKNOWN_INDEX).astype(int)
        )
    return result


def save_category_mapping(mapping: dict, path: str = CATEGORY_MAPPING_FILE) -> None:
    """Simpan mapping kategori ke file JSON supaya indeks konsisten antar refresh data."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(mapping, handle, ensure_ascii=False, indent=2, sort_keys=True)


def load_category_mapping(path: str = CATEGORY_MAPPING_FILE) -> dict:
    """Muat mapping kategori dari file JSON."""
    with open(path, encoding="utf-8") as handle:
        return json.load(handle)


def load_existing_mapping(path: str = CATEGORY_MAPPING_FILE) -> dict:
    """The saved mapping if there is one, otherwise an empty dict.

    Lets build_model_input() extend the numbering across refreshes on a
    machine that already has a mapping, while a first run on a clean checkout
    still builds one from scratch.
    """
    if not Path(path).exists():
        return {}
    return load_category_mapping(path)

## 6. Imputasi nilai kosong

In [6]:
EVENT_PROXIMITY_COLS = [
    "days_into_ramadan",
    "days_until_ramadan",
    "days_since_eid_al_fitr",
    "days_until_eid_al_fitr",
    "days_since_eid_al_adha",
    "days_until_eid_al_adha",
    "days_since_independence_day",
    "days_until_independence_day",
    "days_since_new_year",
    "days_until_new_year",
]


EVENT_PROXIMITY_SENTINEL = 99.0


HISTORY_COLS = [
    "lag_1", "lag_2", "lag_3", "lag_7", "lag_14", "lag_21", "lag_28",
    "roll_mean_7", "roll_std_7", "roll_mean_14", "roll_std_14",
    "roll_mean_28", "roll_std_28",
]


def impute_features(df: pd.DataFrame) -> pd.DataFrame:
    """Fill the nulls a neural net cannot consume, preserving each column's
    meaning. Tree models do not need this, but both adapters run it so the two
    see identical values.
    """
    result = df.copy()

    for col in EVENT_PROXIMITY_COLS:
        result[col] = result[col].fillna(EVENT_PROXIMITY_SENTINEL)

    result["was_relocated"] = result["days_since_relocation"].notna()
    result["days_since_relocation"] = result["days_since_relocation"].fillna(0.0)

    result["has_baseline"] = result["baseline_ratio"].notna()
    result["baseline_ratio"] = result["baseline_ratio"].fillna(1.0)

    # 0 is a legitimate lag value here — 54% of Kuantitas is zero — so filling
    # with it would be indistinguishable from "no demand that day" without an
    # indicator. Both indicators are row-local: how many windows failed to fit
    # is a monotone function of how far into its segment the row sits, so the
    # count doubles as an ordinal measure of available history without needing
    # to group or sort.
    missing = result[HISTORY_COLS].isna()
    result["missing_history_count"] = missing.sum(axis=1).astype(int)
    result["has_full_history"] = result["missing_history_count"] == 0
    for col in HISTORY_COLS:
        result[col] = result[col].fillna(0.0)

    return result

## 7. Adapter tabular & sekuens

In [7]:
FEATURE_COLS = [
    # demand history
    *HISTORY_COLS,
    "has_full_history", "missing_history_count",
    # calendar
    "day_of_week", "day_of_month", "month", "is_weekend", "is_national_holiday",
    "is_ramadan", "days_into_ramadan", "days_until_ramadan",
    "is_eid_al_fitr", "days_since_eid_al_fitr", "days_until_eid_al_fitr",
    "is_eid_al_adha", "days_since_eid_al_adha", "days_until_eid_al_adha",
    "is_independence_day", "days_since_independence_day", "days_until_independence_day",
    "is_new_year", "days_since_new_year", "days_until_new_year",
    # replenishment cycle
    "kawasan", "lead_time_days", "is_delivery_day", "target_window_weekend_days",
    # outlet
    "has_shopee", "has_gofood", "has_grabfood", "can_order_online",
    "branch_avg_daily_qty", "branch_demand_cv", "branch_age_days",
    "days_since_relocation", "was_relocated",
    # item
    "is_event_driven",
    # encoded categoricals
    "Kode Barang_idx", "Nama Cabang_idx", "Kategori Barang_idx", "kota_idx",
    "hari_pengiriman_idx", "branch_volume_tier_idx", "demand_segment_idx",
]


def _resolve_pair_cols(df: pd.DataFrame, pair_cols: Optional[list]) -> list:
    """Group by segment when the frame carries one, otherwise by pair.

    An explicit pair_cols argument always wins. Falling back on the column's
    presence keeps fixtures and callers that predate segmentation working,
    while guaranteeing that any frame built from the segmented panel never
    lets a warm-up cut or an LSTM window bridge a closure.
    """
    if pair_cols is not None:
        return pair_cols
    return SEGMENT_COLS if SEGMENT_COL in df.columns else PAIR_COLS


def drop_warmup_rows(
    df: pd.DataFrame,
    lookback: int = LOOKBACK,
    pair_cols: list = None,
    date_col: str = DATE_COL,
) -> pd.DataFrame:
    """Keep rows whose zero-based position within their own pair is >= lookback.

    These are exactly the rows an LSTM can build a full window for, and exactly
    the rows where lag_28 is non-null.

    This is the first of two cuts both adapters make. The second is the target:
    on the last days of a segment the lead-time window runs past the end of the
    pair's data, so no target exists. Those rows are dropped as *prediction*
    rows but deliberately not removed from the frame — they remain valid
    history inside later windows, and deleting them here would splice the
    series and change what the LSTM sees.
    """
    pair_cols = _resolve_pair_cols(df, pair_cols)
    result = df.sort_values(pair_cols + [date_col]).reset_index(drop=True)
    position = result.groupby(pair_cols, observed=True).cumcount()
    return result[position >= lookback].reset_index(drop=True)


def to_tabular(
    df: pd.DataFrame,
    feature_cols: list,
    target_col: str = TRAIN_TARGET_COL,
    lookback: int = LOOKBACK,
    pair_cols: list = None,
    date_col: str = DATE_COL,
    log_target: bool = False,
) -> dict:
    """Adapter for XGBoost and Random Forest: a flat table, NaNs left in place.

    NaNs are left in the *features* only. Rows with no target are dropped —
    see the note on drop_warmup_rows().

    Pass the same log_target value here and to to_sequences(), or the contract
    check will fail.
    """
    pair_cols = _resolve_pair_cols(df, pair_cols)
    frame = drop_warmup_rows(df, lookback=lookback, pair_cols=pair_cols, date_col=date_col)
    frame = frame[frame[target_col].notna()]
    if log_target:
        frame = frame.copy()
        frame[target_col] = np.log1p(frame[target_col])
    return {
        "X": frame[feature_cols].reset_index(drop=True),
        "y": frame[target_col].reset_index(drop=True),
        "keys": frame[pair_cols + [date_col]].reset_index(drop=True),
        "fold_id": frame["fold_id"].reset_index(drop=True),
    }


def fit_scaler(df: pd.DataFrame, feature_cols: list) -> dict:
    """Per-feature mean and std. Fit on one fold's training rows only —
    fitting globally would leak December statistics into the July fold.
    """
    scaler = {}
    for col in feature_cols:
        mean = float(df[col].mean())
        std = float(df[col].std(ddof=0))
        scaler[col] = (mean, std if std > 0 else 1.0)
    return scaler


def apply_scaler(df: pd.DataFrame, scaler: dict, feature_cols: list) -> pd.DataFrame:
    """Terapkan scaler (mean/std per kolom) ke DataFrame."""
    result = df.copy()
    for col in feature_cols:
        mean, std = scaler[col]
        result[col] = (result[col] - mean) / std
    return result


def save_scaler(scaler: dict, path: str = SCALER_FILE) -> None:
    """Simpan parameter scaler (mean/std) ke JSON supaya bisa dipakai ulang di inference."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    serializable = {col: list(params) for col, params in scaler.items()}
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(serializable, handle, indent=2, sort_keys=True)


def load_scaler(path: str = SCALER_FILE) -> dict:
    """Muat parameter scaler dari JSON."""
    with open(path, encoding="utf-8") as handle:
        return {col: tuple(params) for col, params in json.load(handle).items()}


def inverse_log_target(values: np.ndarray) -> np.ndarray:
    """Undo log1p on predictions. Exact for quantile models: quantiles are
    equivariant under monotonic transforms, so expm1(q_a(log1p(y))) == q_a(y).
    """
    return np.expm1(values)


def to_sequences(
    df: pd.DataFrame,
    feature_cols: list,
    target_col: str = TRAIN_TARGET_COL,
    lookback: int = LOOKBACK,
    pair_cols: list = None,
    date_col: str = DATE_COL,
    log_target: bool = False,
) -> dict:
    """Adapter for the LSTM: one (lookback, n_features) window per predictable
    row, the window ending at that row inclusive.

    Produces exactly the rows drop_warmup_rows() keeps, so to_tabular() and
    to_sequences() agree — see validate_contract(). Pass the same log_target
    value to both adapters or the contract check will fail.
    """
    pair_cols = _resolve_pair_cols(df, pair_cols)
    frame = df.sort_values(pair_cols + [date_col]).reset_index(drop=True)
    if log_target:
        frame = frame.copy()
        frame[target_col] = np.log1p(frame[target_col])

    windows, targets, key_rows, folds = [], [], [], []
    for _, group in frame.groupby(pair_cols, observed=True, sort=False):
        values = group[feature_cols].to_numpy(dtype="float32")
        target_values = group[target_col].to_numpy(dtype="float32")
        fold_values = group["fold_id"].to_numpy()
        keys = group[pair_cols + [date_col]].to_numpy()

        for position in range(lookback, len(group)):
            if np.isnan(target_values[position]):
                continue
            windows.append(values[position - lookback + 1 : position + 1])
            targets.append(target_values[position])
            key_rows.append(keys[position])
            folds.append(fold_values[position])

    if windows:
        stacked = np.stack(windows).astype("float32")
    else:
        stacked = np.empty((0, lookback, len(feature_cols)), dtype="float32")

    return {
        "X": stacked,
        "y": np.asarray(targets, dtype="float32"),
        "keys": pd.DataFrame(key_rows, columns=pair_cols + [date_col]),
        "fold_id": pd.Series(folds, dtype="float64"),
    }


def validate_contract(tabular: dict, sequences: dict, require_finite: bool = True) -> None:
    """Guarantee the two adapters expose the same rows, targets, and folds.

    Without this, "the LSTM is 8% better" could really mean "the LSTM was
    evaluated on a different 5% of the rows".

    require_finite also rejects NaN in either feature block. Matching rows are
    not enough on their own: a tree model consumes NaN natively while an LSTM
    turns it into NaN loss, so a tensor that still carries nulls gets patched
    at training time and the two models silently stop seeing the same inputs.
    Pass False only for a run that is not comparing against the LSTM.
    """
    if require_finite:
        tabular_nan = int(np.isnan(np.asarray(tabular["X"], dtype="float64")).sum())
        assert tabular_nan == 0, (
            f"Fitur tabular mengandung {tabular_nan} NaN — jalankan impute_features()"
        )
        sequence_nan = int(np.isnan(np.asarray(sequences["X"], dtype="float64")).sum())
        assert sequence_nan == 0, (
            f"Tensor sequence mengandung {sequence_nan} NaN — jalankan impute_features()"
        )

    tabular_keys = tabular["keys"].reset_index(drop=True)
    sequence_keys = sequences["keys"].reset_index(drop=True)

    assert len(tabular_keys) == len(sequence_keys), (
        f"Adapter menghasilkan jumlah baris berbeda: "
        f"tabular {len(tabular_keys)}, sequence {len(sequence_keys)}"
    )

    tabular_set = set(map(tuple, tabular_keys.to_numpy()))
    sequence_set = set(map(tuple, sequence_keys.to_numpy()))
    assert tabular_set == sequence_set, (
        f"Adapter menghasilkan baris berbeda: "
        f"{len(tabular_set - sequence_set)} hanya di tabular, "
        f"{len(sequence_set - tabular_set)} hanya di sequence"
    )

    tabular_y = np.asarray(tabular["y"], dtype="float64")
    sequence_y = np.asarray(sequences["y"], dtype="float64")

    # Checked regardless of require_finite: a NaN feature is a modelling choice
    # (XGBoost consumes it natively), a NaN label is not. Random Forest raises
    # on it, an LSTM turns it into NaN loss, and equal_nan below would let two
    # NaNs compare equal — so without this the adapters can agree on a target
    # that no model can actually train against.
    label_nan = int(np.isnan(tabular_y).sum() + np.isnan(sequence_y).sum())
    assert label_nan == 0, (
        f"Ditemukan {label_nan} target NaN — baris tanpa target tidak bisa "
        "dilatih maupun dinilai dan harus dibuang oleh adapter"
    )

    assert np.allclose(tabular_y, sequence_y, equal_nan=True), (
        "Nilai target berbeda antar adapter"
    )

    tabular_fold = np.asarray(tabular["fold_id"], dtype="float64")
    sequence_fold = np.asarray(sequences["fold_id"], dtype="float64")
    assert np.allclose(tabular_fold, sequence_fold, equal_nan=True), (
        "Pembagian fold berbeda antar adapter"
    )

## 8. Urutan langkah build model input

In [8]:
def build_model_input(
    featured_path: str = FEATURED_FILE,
    event_items_path: str = EVENT_ITEMS_FILE,
    cutoff: pd.Timestamp = TEST_START,
    mapping_path: str = CATEGORY_MAPPING_FILE,
) -> pd.DataFrame:
    df = pd.read_parquet(featured_path)
    df = add_event_flag(df, load_event_items(event_items_path))
    df = classify_pairs(df, cutoff=cutoff)
    df = assign_folds(df)
    df = impute_features(df)

    mapping = build_category_mapping(
        df, cutoff=cutoff, existing=load_existing_mapping(mapping_path)
    )
    save_category_mapping(mapping, mapping_path)
    df = encode_categoricals(df, mapping)
    return df

## 9. Jalankan build model input

In [9]:
model_input = build_model_input()

print(f"{len(model_input):,} baris × {len(model_input.columns)} kolom")
model_input[["is_event_driven", "demand_segment", "fold_id"]].head()

1,502,522 baris × 82 kolom


,is_event_driven,demand_segment,fold_id
0,False,smooth,NaN
1,False,smooth,NaN
2,False,smooth,NaN
3,False,smooth,NaN
4,False,smooth,NaN


## 10. QA — segmen & fold

In [10]:
pair_segment = model_input.groupby(PAIR_COLS, observed=True)["demand_segment"].first()
print(pair_segment.value_counts(), "\n")
print(model_input["fold_id"].value_counts(dropna=False).sort_index())

assert model_input.loc[model_input["Tanggal"] >= "2025-12-01", "fold_id"].isna().all(), \
    "Baris Desember tidak boleh punya fold — itu test set terkunci"
print("\n✓ Desember bebas fold")

demand_segment
intermittent    1304
lumpy            943
erratic          407
smooth           325
Name: count, dtype: int64 

fold_id
1.0      76263
2.0      76266
3.0      70982
4.0      69392
5.0      61165
NaN    1148454
Name: count, dtype: int64

✓ Desember bebas fold


## 11. QA — imputasi

In [11]:
# Cek yang paling penting: kolom kedekatan event tidak boleh terisi 0 (itu berarti "hari ini
# Idul Fitri"), dan sentinel harus di atas 70 — days_until_ramadan mencapai 70 di data asli.
for col in EVENT_PROXIMITY_COLS:
    assert model_input[col].notna().all(), f"{col} masih punya null"

filled = (model_input["days_until_ramadan"] == EVENT_PROXIMITY_SENTINEL).mean()
print(f"days_until_ramadan sentinel: {filled:.1%} baris")
assert EVENT_PROXIMITY_SENTINEL > 70, "Sentinel bertabrakan dengan nilai asli"

print(model_input[["was_relocated", "has_baseline"]].mean().round(3))
print("\n✓ imputasi aman")

days_until_ramadan sentinel: 84.5% baris
was_relocated    0.157
has_baseline     0.856
dtype: float64

✓ imputasi aman


## 12. QA — kontrak adapter

In [12]:
# Ini yang membuat perbandingan XGBoost / Random Forest / LSTM bisa dipertanggungjawabkan:
# kedua adapter wajib melihat baris yang sama persis.
feature_cols = [
    "lag_1", "lag_7", "lag_28", "roll_mean_7", "roll_mean_28", "roll_std_7",
    "day_of_week", "is_weekend", "is_national_holiday", "lead_time_days",
    "days_until_ramadan", "days_since_relocation", "was_relocated",
    "baseline_ratio", "has_baseline", "is_event_driven",
    "Kode Barang_idx", "Nama Cabang_idx", "kota_idx", "demand_segment_idx",
]

sample_branches = model_input["Nama Cabang"].unique()[:3]
sample = model_input[model_input["Nama Cabang"].isin(sample_branches)]

tabular = to_tabular(sample, feature_cols=feature_cols)
sequences = to_sequences(sample, feature_cols=feature_cols)
validate_contract(tabular, sequences)

print(f"tabular X : {tabular['X'].shape}")
print(f"sequence X: {sequences['X'].shape}")
print("✓ kontrak lolos")

tabular X : (97345, 20)
sequence X: (97345, 28, 20)
✓ kontrak lolos


## 13. Export model_input.parquet

In [13]:
def export_model_input(df: pd.DataFrame, path: str = MODEL_INPUT_FILE) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)

In [14]:
export_model_input(model_input, MODEL_INPUT_FILE)
print(f"✓ ditulis ke {MODEL_INPUT_FILE}")

✓ ditulis ke /Users/ramapdp/Project/Personal/forecast-scm/dataset/model_ready/model_input.parquet


## 14. *(Opsional)* Cek sinkron dengan `utils/`

In [15]:
import inspect
import re
import sys

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from utils.modelling import modeling_prep as _ref_prep
from utils.modelling import purging as _ref_purging

_MODUL_REF = [_ref_prep, _ref_purging]
_QUALIFIER = re.compile(r"\b(modeling_prep|build_panel|purging)\.")
# Nama yang memang sengaja beda dari utils, jadi tidak dibandingkan.
_LEWATI = {"find_base_dir", "_MODUL_REF", "_QUALIFIER", "_LEWATI", "_kode",
           "_rujukan", "_nilai_sama", "BASE_DIR", "MODEL_READY_DIR", "FEATURED_FILE", "MODEL_INPUT_FILE",
           "EVENT_ITEMS_FILE", "CATEGORY_MAPPING_FILE", "SCALER_FILE"}


def _kode(fn) -> str:
    """Kode sumber fungsi tanpa qualifier modul — di notebook semua fungsi satu namespace."""
    return _QUALIFIER.sub("", inspect.getsource(fn))


def _rujukan(nama):
    """Objek bernama `nama` di modul utils mana pun; None kalau namanya bukan milik utils."""
    for modul in _MODUL_REF:
        if hasattr(modul, nama):
            return getattr(modul, nama)
    return None


beda, n_fungsi, n_konstanta = [], 0, 0
for nama, obj in sorted(globals().items()):
    if nama in _LEWATI or nama.startswith("__"):
        continue
    ref = _rujukan(nama)
    if ref is None:
        continue
    if inspect.isfunction(obj):
        n_fungsi += 1
        if _kode(obj) != _kode(ref):
            beda.append(nama)
    elif nama.isupper() or nama.startswith("_"):
        n_konstanta += 1
        if obj != ref:
            beda.append(nama)

if beda:
    print("BERBEDA dari utils/ — salin ulang atau samakan: " + ", ".join(beda))
else:
    print(f"Sinkron: {n_fungsi} fungsi + {n_konstanta} konstanta identik dengan utils/modelling/")

Sinkron: 26 fungsi + 18 konstanta identik dengan utils/modelling/
